In [ ]:
import numpy as np
import pandas as pd
import gymnasium as gym
import gym_trading_env
from stable_baselines3 import DQN
from gym_trading_env.wrapper import DiscreteActionsWrapper

#DYNAMIQUE FEATURE

import pandas_ta as ta

def preprocess(df):
    df.ta.mfi(length=14, append=True) 
    df['MFI_14'] = df['MFI_14'] / 100.0
    df.fillna(50, inplace=True)
    return df

def reward_function(history):   
    current_val = history["portfolio_valuation", -1]
    last_val = history["portfolio_valuation", -2]
    reward = (current_val / last_val) - 1
    
    return reward

base_env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess,
    portfolio_initial_value=1_000,
    trading_fees=0.1/100,
    borrow_interest_rate=0.02/100/24,
    reward_function=reward_function,
)

env = DiscreteActionsWrapper(base_env, positions=[-1, 0, 1, 2])
obs, _ = env.reset()
obs, reward, terminated, truncated, info = env.step(3)
print("obs :", obs)
print("reward :", reward)
print("terminated :", terminated)
print("truncated :", truncated)
print("info :", info)

model = DQN(
    "MlpPolicy", 
    env, 
    verbose=1, 
    learning_rate=0.00001,
    buffer_size=100000,
    batch_size=32,
    exploration_fraction=0.2, # Explore during 50% of training
    exploration_initial_eps=1.0, # Start with 100% exploration
    exploration_final_eps=0.05,
)
model.learn(total_timesteps = 300000, log_interval = 4)
model.save("dqn_test")


In [ ]:
nb_episodes = 10
for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    print(f'Episode n˚{episode}')
    done = False

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(int(action))
        done = terminated or truncated

    if terminated:
        print('Argent perdu')
    elif truncated:
        print('Épisode terminé')
    env.unwrapped.save_for_render(dir = "render_logs")



In [1]:
import numpy as np
import pandas as pd
import gymnasium as gym
import gym_trading_env
from gym_trading_env.wrapper import DiscreteActionsWrapper
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
import pandas_ta as ta
import wandb
from wandb.integration.sb3 import WandbCallback

def preprocess(df):
    df["log_ret"] = np.log(df["close"]).diff()

    # Tendance
    df.ta.macd(append=True)
    df.ta.ema(length=20, append=True)
    df.ta.ema(length=50, append=True)
    
    # Calcul de distance par rapport aux EMA (normalisation)
    df["dist_ema20"] = (df["close"] - df["EMA_20"]) / df["EMA_20"]
    
    # Oscillateurs / Momentum
    df.ta.rsi(length=14, append=True)
    df["RSI_14"] = df["RSI_14"] / 100.0 

    # Volatilité
    df.ta.atr(length=14, append=True)
    df["ATR_14_norm"] = df["ATRr_14"] / df["close"] 

    # Nettoyage
    df.dropna(inplace=True) 
    return df

def reward_function(history):
    # Log return du portefeuille
    return np.log(history["portfolio_valuation", -1] / history["portfolio_valuation", -2])

def metric_portfolio_valuation(history):
    return round(history['portfolio_valuation', -1], 2)

# Création de l'environnement
base_env = gym.make(
    "MultiDatasetTradingEnv",
    dataset_dir="data/*.pkl",
    preprocess=preprocess,
    portfolio_initial_value=1_000,
    trading_fees=0.1/100,
    borrow_interest_rate=0.02/100/24,
    reward_function=reward_function,
)

base_env.add_metric('Portfolio Valuation', metric_portfolio_valuation)

env = DiscreteActionsWrapper(base_env, positions=[-1, 0, 1, 2])
#env = DummyVecEnv([lambda: env])

# Modèle PPO
model = PPO(
    "MlpPolicy", 
    env, 
    verbose=1, 
    n_steps=2048,
    ent_coef=0.01,
    learning_rate=0.00001,
    batch_size=32,
)

print("Début de l'entraînement...")
#300_000 ça marche pas mal 
model.learn(
    total_timesteps=1_000_000, log_interval = 4
)

portfolio_valuation = base_env.historical_info['portfolio_valuation', -1]

model.save("ppo_trading_final")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Début de l'entraînement...
Market Return : 42.46%   |   Portfolio Return : -98.49%   |   Portfolio Valuation : 15.08   |   
Market Return : 26.64%   |   Portfolio Return : -99.50%   |   Portfolio Valuation : 4.99   |   
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 3.95e+03      |
|    ep_rew_mean          | -4.75         |
| time/                   |               |
|    fps                  | 623           |
|    iterations           | 4             |
|    time_elapsed         | 13            |
|    total_timesteps      | 8192          |
| train/                  |               |
|    approx_kl            | 0.00019141959 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -1.39         |
|    explained_variance   | -0.315        |
|    learning_rate        

In [2]:
nb_episodes = 10
for episode in range(1, nb_episodes + 1):
    obs, _ = env.reset()
    print(f'Episode n˚{episode}')
    done = False

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(int(action))
        done = terminated or truncated

    if terminated:
        print('Argent perdu')
    elif truncated:
        print('Épisode terminé')
    env.unwrapped.save_for_render(dir = "render_logs")



Episode n˚1
Market Return : 699.56%   |   Portfolio Return : 1396.99%   |   Portfolio Valuation : 14969.94   |   
Épisode terminé
Episode n˚2
Market Return : 959.21%   |   Portfolio Return : 1916.02%   |   Portfolio Valuation : 20160.24   |   
Épisode terminé
Episode n˚3
Market Return :  9.68%   |   Portfolio Return : 19.12%   |   Portfolio Valuation : 1191.19   |   
Épisode terminé
Episode n˚4
Market Return :  9.68%   |   Portfolio Return : 19.20%   |   Portfolio Valuation : 1191.98   |   
Épisode terminé
Episode n˚5
Market Return : 104.07%   |   Portfolio Return : 207.78%   |   Portfolio Valuation : 3077.79   |   
Épisode terminé
Episode n˚6
Market Return : 26.64%   |   Portfolio Return : 53.10%   |   Portfolio Valuation : 1530.99   |   
Épisode terminé
Episode n˚7
Market Return : 42.46%   |   Portfolio Return : 84.65%   |   Portfolio Valuation : 1846.48   |   
Épisode terminé
Episode n˚8
Market Return : 39.26%   |   Portfolio Return : 78.20%   |   Portfolio Valuation : 1782.01   |  